# P6 evaluation: guided usage

Read [the concrete plan](../docs/P6/EVALUATION_EXECUTION_PLAN.md) and [team handoff](../docs/P6/EVALUATION_TEAM_HANDOFF.md). This notebook uses the real implemented CLI. It runs only synthetic, offline examples by default: no downloads, API calls, human decisions or final inference.

**Deliverables:** audited gold labels → retrieval table; saved answers/references → QA table; exact contexts + configured judge → RAGAS table; timings → performance table; matched scores → comparison table.

Run top to bottom in the existing project Python kernel (Python 3.10+, Pydantic 2). Reusable code is in `src/evaluation/`. All sample numbers are teaching examples, not project results. Clear outputs before sharing. Schemas and split mappings remain `v0-proposed`.


## 1. Check setup

**Action:** run this cell from inside the repository. **Expected:** checkout details and CLI usage. No model is loaded.


In [ ]:
from pathlib import Path
import subprocess, sys, tempfile, json, math
ROOT = Path.cwd().resolve()
while ROOT != ROOT.parent and not (ROOT/'.git').exists():
    ROOT = ROOT.parent
if not (ROOT/'.git').exists():
    raise RuntimeError('Start the notebook inside the repository')
def cli(*args):
    result = subprocess.run([sys.executable, '-m', 'scripts.run_p6_evaluation', *map(str,args)],
                            cwd=ROOT, text=True, capture_output=True)
    if result.returncode:
        raise RuntimeError(result.stderr)
    print(result.stdout)
print('Python:', sys.executable)
print('Commit:', subprocess.check_output(['git','-C',str(ROOT),'rev-parse','HEAD'],text=True).strip())
cli('--help')


## 2. Understand the metrics

| Metric | Calculation | Example expectation |
| --- | --- | --- |
| Precision@3 | Relevant returned / 3 | A,B,C with relevant A,C,D → 2/3 |
| Pool recall@3 | Relevant returned / relevant pool | Same example → 2/3 |
| MRR@3 | 1 / first relevant rank | A is first → 1 |
| nDCG@3 | Discounted relevance / ideal discount | Same example → about .704 |
| QA token F1 | Harmonic mean of overlap precision/recall | “BERT model” vs “BERT base model” → .8 |
| Agreement | Initial matching judgments / double-labeled pairs | 90 matches / 100 → .9 |

RAGAS faithfulness checks claims against actual answer evidence; response relevance checks whether an answer addresses the question; context precision evaluates useful evidence placement. These need a configured judge, not just arithmetic. Undefined/failed metrics retain status codes. See the [metric reference](../docs/P6/EVALUATION_METRICS_REFERENCE.md).


In [ ]:
from src.evaluation.qa import pair_scores
from src.evaluation.agreement import agreement
assert pair_scores('BERT model','BERT base model')['f1'] == .8
sample = agreement([(0,0)]*45 + [(1,1)]*45 + [(0,1)]*5 + [(1,0)]*5)
assert math.isclose(sample['agreement'], .9)
assert math.isclose(sample['kappa'], .8)
print('Worked examples passed:', pair_scores('BERT model','BERT base model'), sample)


## 3. Run the implemented QA scorer

**Action:** execute both synthetic arms. **Expected:** static answerable F1=.8, adaptive=1; both correctly refuse the unanswerable question. The tool saves scores, CSV summary and a provenance manifest. Examples use a temporary directory and never overwrite real results. The printed path remains available while this kernel session is active; save real results under ignored `results/p6/` using the handoff commands.


In [ ]:
demo = tempfile.TemporaryDirectory(prefix='p6-evaluation-example-')
OUT = Path(demo.name)
FIX = ROOT/'tests/fixtures/p6'
for arm in ('static','adaptive'):
    cli('qa','--questions',FIX/'questions.jsonl','--predictions',FIX/(arm+'.jsonl'),
        '--references',FIX/'references.jsonl','--arm',arm,'--role','development',
        '--output',OUT/(arm+'-qa'))
    summary = json.loads((OUT/(arm+'-qa')/'summary.json').read_text())
    print(arm, 'F1:',summary['answerable_f1'], 'unanswerable:',summary['unanswerable_correctness'])
assert json.loads((OUT/'static-qa/summary.json').read_text())['answerable_f1'] == .8
assert json.loads((OUT/'adaptive-qa/summary.json').read_text())['answerable_f1'] == 1.


## 4. Validate RAGAS inputs without calling a judge

**Action:** run the implemented preparation/validation path. **Expected:** eligible scores are null with `ready_not_scored`; the refusal remains a separate status. This proves file preparation, not live evaluator quality. Do not replace null scores with zero or one.


In [ ]:
cli('ragas','--questions',FIX/'questions.jsonl','--predictions',FIX/'static.jsonl',
    '--references',FIX/'references.jsonl','--contexts',FIX/'contexts.jsonl',
    '--arm','static','--role','development','--output',OUT/'ragas-validation')
print((OUT/'ragas-validation/scores.jsonl').read_text())


## 5. Performance and a matched comparison

**Action:** calculate client timing and compare QA scores. **Expected:** performance has two timed synthetic outcomes; comparison is right(adaptive) minus left(static). F1 pairs only the answerable question; task_score also includes unanswerable correctness. Five or fewer examples are not a stable p95 benchmark.

`elapsed_s + answer_s` is internal pipeline time. `client_elapsed_s` is a separately measured HTTP duration. Throughput requires measured batch wall time, not the sum of concurrent spans. Synthetic price values are not real prices.


In [ ]:
cli('performance','--predictions',FIX/'static.jsonl','--arm','static',
    '--latency-field','client_elapsed_s','--batch-seconds','6.5',
    '--hardware','synthetic-example','--cache-policy','no-result-cache',
    '--output',OUT/'performance')
cli('compare','--left',OUT/'static-qa/scores.jsonl','--right',OUT/'adaptive-qa/scores.jsonl',
    '--metrics','f1','task_score','--resamples','1000','--seed','42','--output',OUT/'comparison')
print((OUT/'performance/performance.csv').read_text())
print((OUT/'comparison/ablation.csv').read_text())


## 6. Process P3 labels: real next action

Use handoff sections 3–5. No real-label processing is automatically executed in this notebook.

1. `audit-labels --ref <reviewed commit> --output <new directory>` validates sheets and produces agreement plus an adjudication CSV.
2. Humans review submission independence/history. The inspected 4 November commit has 1,665/1,853 matching labels (89.854%) and 188 disagreements; older snapshots differ. Method names were visible.
3. A third person resolves each disagreement with final 0/1 label and reason, leaving original fields unchanged.
4. `finalize-labels` validates actual review, exact corpus identity and adjudication. It exports `final_labels.jsonl`, original judgments and a compatible manifest.
5. `export-p3` exports pinned existing questions/pairs/rankings. `retrieval` scores K=1,3,5 without running models. All rankings contributed to the judged pool; pool recall is not exhaustive paper recall.

The older `finalize_annotation_gold.ipynb` is useful explanation, but its historical snapshot and draft manifest should not be treated as current final scoring artifacts. Use the new CLI's completed export contract. Never set review flags just to make a command run.


## 7. Real model outputs and live RAGAS

P4/P5 must supply saved predictions, reference linkage, original prompt configuration and timings. P3 relevance labels cannot substitute for QA reference answers. ID joins use `question_id`, never row order; P5 request-only files need a verified mapping.

Ask for the exact ordered/truncated answer contexts. `contexts` can reconstruct them only using the reviewed original P5 configuration, a matching prompt version and the original agent inputs. Critic context and every retrieved passage are not the same as answer context.

Use the separate judge environment and configuration in handoff section 8. First validate inputs without `--live`, then smoke-test a small development set with `--live`. No credentials belong in a notebook or Git. A live judge has not been verified in this implementation session. Refusals, missing context/reference and failed judge calls remain explicit statuses with denominators.

The existing agent runner refuses the official test split. Final runs need a reviewed freeze and named operator; do not bypass its guard through `--items`. Freeze prompts/model/retrieval/context budget/metrics before final inference; join references only afterward for scoring.


## 8. Completion expectations

| Deliverable | Finish when |
| --- | --- |
| Gold labels | Initial agreement preserved, actual independence reviewed, third-person adjudication complete, corpus hash validated |
| Retrieval table | Pinned rankings/gold matched; pool metrics and coverage clearly labeled |
| QA table | Correct reference/answer-type mapping, one expected outcome per question, exclusions/errors/refusals disclosed |
| RAGAS table | Real judge smoke-tested, exact contexts verified, model/version/configuration and valid counts saved |
| Performance | Hardware/concurrency/cache policy fixed; timing scope, counts, real wall time and known usage stated |
| Comparison | Same frozen questions/settings, matched denominators and paper-clustered uncertainty |
| Handoff | Input/output hashes, commands, dependency snapshots and owner-reviewed interpretations available |

Static/adaptive compares the combined module. Separate rewriting/reranking claims need a 2×2 study; larger-depth static controls distinguish smarter retrieval from simply reading more text. Keep final real artifacts in ignored `results/` and share through the approved external archive. Clear notebook outputs before committing.


## 9. Static depths and adaptive evidence recall
The implemented `evidence` command uses P3 mapped passage IDs, separately from pool labels. This synthetic example expects recall 1/3, 2/3, 1 at static depths 5/8/10. Adaptive union across rounds reaches 1. It does not imply those results on real questions. See team handoff section 12 for full CLI commands and held-out coordination for freeze steps.


In [ ]:
from src.evaluation.evidence import score_evidence
questions = [{'question_id':'example','paper_id':'paper','question':'Example?'}]
corpus = [{'passage_id':str(i),'paper_id':'paper'} for i in range(1,11)]
gold = [{'question_id':'example','paper_id':'paper','gold_passage_ids':['1','8','10']}]
static = [{'question_id':'example','retrieved_passage_ids':[str(i) for i in range(1,11)]}]
rows, summary = score_evidence(questions,static,gold,corpus,'static','static',[5,8,10],10)
assert [r['mapped_recall'] for r in rows] == [1/3,2/3,1]
adaptive = [{'question_id':'example','retrieval_log':[{'round':1,'passage_ids':['1']}, {'round':2,'passage_ids':['8','10']}]}]
a_rows, _ = score_evidence(questions,adaptive,gold,corpus,'adaptive','adaptive')
assert a_rows[0]['mapped_recall'] == 1
print('Static recall:', [(r['k'],r['mapped_recall']) for r in rows])
print('Adaptive cumulative recall:', a_rows[0]['mapped_recall'])
